# Assignment 7.2 - XGBoost



## Task 7.2.1: XGBoost - Regression

* Build an XGBoost classifier using `numpy` only. Train your XGBoost model on the `California Housing` regression task. Report on the performance predicting unseen test samples. **(RESULTS)**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Class structure that might help. Feel free to modify as needed.
class TreeNode:
    """Represents a node in the decision tree"""
    def __init__(self, depth=0):
        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None  # Leaf value (w_j*)
        self.is_leaf = False
        self.depth = depth
        self.indices = None # Store the indices of samples belonging to this node
        self.G = None       # Sum of gradients for samples in this node
        self.H = None       # Sum of hessians for samples in this node


class DecisionTree:
    """Decision tree for XGBoost (Weak Learner)"""
    def __init__(self, max_depth, min_samples_split, gamma, reg_lambda):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.gamma = gamma
        self.reg_lambda = reg_lambda
        self.root = None

    def _calculate_leaf_value(self, G, H):
        """Calculate the optimal leaf weight (w_j*)"""
        return -G / (H + self.reg_lambda)

    def _calculate_gain(self, G_L, H_L, G_R, H_R):
        """Calculate the loss reduction (Gain) for a split"""

        score_L = G_L**2 / (H_L + self.reg_lambda)
        score_R = G_R**2 / (H_R + self.reg_lambda)

        G_parent = G_L + G_R
        H_parent = H_L + H_R
        score_parent = G_parent**2 / (H_parent + self.reg_lambda)

        gain = 0.5 * (score_L + score_R - score_parent) - self.gamma
        return gain

    def _find_best_split(self, X, gradients, hessians, indices):
        """Find the best feature and threshold to split the data"""
        X_subset = X[indices]
        gradients_subset = gradients[indices]
        hessians_subset = hessians[indices]

        best_gain = -np.inf
        best_feature_index = None
        best_threshold = None
        best_left_indices = None
        best_right_indices = None

        n_samples, n_features = X_subset.shape

        if n_samples < self.min_samples_split:
            return None, None, None, None, None

        for feature_index in range(n_features):
            unique_values = np.unique(X_subset[:, feature_index])

            for threshold in unique_values:
                is_left = X_subset[:, feature_index] <= threshold

                left_indices = indices[is_left]
                right_indices = indices[~is_left]

                if len(left_indices) < self.min_samples_split or len(right_indices) < self.min_samples_split:
                    continue

                G_L = np.sum(gradients[left_indices])
                H_L = np.sum(hessians[left_indices])
                G_R = np.sum(gradients[right_indices])
                H_R = np.sum(hessians[right_indices])

                current_gain = self._calculate_gain(G_L, H_L, G_R, H_R)

                if current_gain > best_gain:
                    best_gain = current_gain
                    best_feature_index = feature_index
                    best_threshold = threshold
                    best_left_indices = left_indices
                    best_right_indices = right_indices

        if best_gain > self.gamma:
            return best_feature_index, best_threshold, best_left_indices, best_right_indices, best_gain
        else:
            return None, None, None, None, None

    def _build_tree(self, X, gradients, hessians, indices, depth):
        """Recursive function to build the tree"""

        node = TreeNode(depth=depth)
        node.indices = indices

        node.G = np.sum(gradients[indices])
        node.H = np.sum(hessians[indices])

        if depth >= self.max_depth or len(indices) < self.min_samples_split:
            node.is_leaf = True
            node.value = self._calculate_leaf_value(node.G, node.H)
            return node

        feature_index, threshold, left_indices, right_indices, gain = self._find_best_split(
            X, gradients, hessians, indices
        )

        if feature_index is None: # No good split found or gain too low
            node.is_leaf = True
            node.value = self._calculate_leaf_value(node.G, node.H)
            return node

        node.feature_index = feature_index
        node.threshold = threshold

        node.left = self._build_tree(X, gradients, hessians, left_indices, depth + 1)
        node.right = self._build_tree(X, gradients, hessians, right_indices, depth + 1)

        return node

    def fit(self, X, gradients, hessians):
        """Build the tree"""
        all_indices = np.arange(X.shape[0])
        self.root = self._build_tree(X, gradients, hessians, all_indices, depth=0)

    def _traverse_tree(self, x, node):
        """Traverse the tree for a single sample x"""
        if node.is_leaf:
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

    def predict(self, X):
        """Predict using the tree for all samples"""
        if self.root is None:
            return np.zeros(X.shape[0])

        predictions = np.array([self._traverse_tree(x, self.root) for x in X])
        return predictions

In [ ]:
class XGBoost:
    """XGBoost implementation - Supports Regression and Binary Classification"""

    def __init__(
        self,
        task='regression',
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        min_samples_split=2,
        gamma=0.0,
        reg_lambda=1.0
    ):
        self.task = task
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.gamma = gamma
        self.reg_lambda = reg_lambda
        self.trees = []
        self.initial_prediction = None

    def _calculate_gh(self, y, F_m):
        """Calculate gradient (g) and hessian (h) based on the task"""
        if self.task == 'regression':
            # Loss: L(y, F) = 0.5 * (y - F)^2 (Squared Error)
            # g_i = F_m - y (Residual)
            # h_i = 1 (Constant)
            gradients = F_m - y
            hessians = np.ones_like(y)
            return gradients, hessians

        elif self.task == 'classification':
            # Loss: Binary Log Loss (Cross-Entropy)
            # Probability p = 1 / (1 + exp(-F_m))
            # g_i = p - y
            # h_i = p * (1 - p)
            p = 1.0 / (1.0 + np.exp(-F_m))
            gradients = p - y
            hessians = p * (1 - p)
            return gradients, hessians
        else:
            raise ValueError("Task must be 'regression' or 'classification'")

    def fit(self, X, y):
        """Train the XGBoost model"""
        self.trees = []

        if self.task == 'regression':
            # Initial prediction F_0 is the mean of the target y
            self.initial_prediction = np.mean(y)
            # F_m is initialized to F_0
            F_m = np.full_like(y, self.initial_prediction, dtype=float)

        elif self.task == 'classification':
            # Initial prediction F_0 is the log-odds of the average target value
            # F_0 = log(p_0 / (1-p_0)), where p_0 = mean(y)
            p_0 = np.mean(y)
            # Clamp p_0 to avoid log(0) or log(inf)
            p_0 = np.clip(p_0, 1e-15, 1 - 1e-15)
            self.initial_prediction = np.log(p_0 / (1.0 - p_0))
            # F_m is initialized to F_0
            F_m = np.full_like(y, self.initial_prediction, dtype=float)

        # Boosting iterations
        for m in range(self.n_estimators):

            # 1. Calculate gradients (g) and hessians (h)
            gradients, hessians = self._calculate_gh(y, F_m)

            # 2. Fit a new tree (DecisionTree) on the (X, g, h) data
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                gamma=self.gamma,
                reg_lambda=self.reg_lambda
            )
            tree.fit(X, gradients, hessians)

            # 3. Get predictions (phi_m) from the new tree
            phi_m = tree.predict(X)

            # 4. Update the overall prediction F_m
            # F_{m+1} = F_m + eta * phi_m
            F_m += self.learning_rate * phi_m

            # 5. Store the tree
            self.trees.append(tree)

    def predict(self, X):
        """Make predictions (raw score F)"""
        # Start with the initial prediction
        F_m = np.full(X.shape[0], self.initial_prediction, dtype=float)

        # Aggregate predictions from all trees
        for tree in self.trees:
            F_m += self.learning_rate * tree.predict(X)

        if self.task == 'regression':
            return F_m
        elif self.task == 'classification':
            # For classification, return the predicted class (0 or 1)
            probabilities = 1.0 / (1.0 + np.exp(-F_m))
            return (probabilities >= 0.5).astype(int)
        else:
            return F_m

    def predict_proba(self, X):
        """Predict probabilities for binary classification"""
        if self.task != 'classification':
            raise ValueError("predict_proba is only available for 'classification' task.")

        # Get the raw score F
        F_m = np.full(X.shape[0], self.initial_prediction, dtype=float)
        for tree in self.trees:
            F_m += self.learning_rate * tree.predict(X)

        # Convert raw score F (log-odds) to probability p using the sigmoid function
        probabilities = 1.0 / (1.0 + np.exp(-F_m))

        # Return probability of [Class 0, Class 1]
        return np.column_stack([1.0 - probabilities, probabilities])


In [ ]:

from sklearn.datasets import fetch_california_housing

# Load California Housing data
data = fetch_california_housing()
X = data.data
y = data.target

# TODO: Continue here...

# 1. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Standardize the features (important for many ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Instantiate and train the XGBoost Regressor
# Hyperparameters chosen for a reasonable trade-off between speed and performance.
xgb_regressor = XGBoost(
    task='regression',
    n_estimators=100,      # Number of boosting rounds (trees)
    learning_rate=0.1,     # Shrinkage (eta)
    max_depth=4,           # Maximum depth of a tree
    min_samples_split=5,   # Minimum samples required to split a node
    gamma=0.1,             # Minimum loss reduction to make a split
    reg_lambda=1.0         # L2 regularization term
)

print("Starting XGBoost Regression Training...")
xgb_regressor.fit(X_train_scaled, y_train)
print("Training Complete.")

# 4. Make predictions on the test set
y_pred = xgb_regressor.predict(X_test_scaled)

# 5. Evaluate and report results
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- RESULTS: California Housing Regression ---")
print(f"XGBoost Model Parameters:")
print(f"  - Estimators: {xgb_regressor.n_estimators}")
print(f"  - Learning Rate: {xgb_regressor.learning_rate}")
print(f"  - Max Depth: {xgb_regressor.max_depth}")
print(f"Test Root Mean Squared Error (RMSE): {rmse:.4f}")
print("--------------------------------------------------")

Starting XGBoost Regression Training...
Training Complete.

--- RESULTS: California Housing Regression ---
XGBoost Model Parameters:
  - Estimators: 100
  - Learning Rate: 0.1
  - Max Depth: 4
Test Root Mean Squared Error (RMSE): 0.5061
--------------------------------------------------


## Task 7.2.2: XGBoost - Classification (BONUS)

* Train an XGBoost model on the `Breast Cancer` binary classification task. Report on the performance predicting unseen test samples. **(RESULTS)**

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load the dataset
data = load_breast_cancer()

# Access the features and labels
X = data.data  # Shape: (569, 30)
y = data.target  # Shape: (569,) - 0 for malignant, 1 for benign


# 1. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Instantiate and train the XGBoost Classifier
xgb_classifier = XGBoost(
    task='classification', # Switches to Log-Loss and uses p(1-p) for Hessian
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=2,
    gamma=0.0,
    reg_lambda=1.0
)

print("\nStarting XGBoost Classification Training (BONUS)...")
xgb_classifier.fit(X_train_scaled, y_train)
print("Training Complete.")

# 4. Make predictions on the test set
y_pred = xgb_classifier.predict(X_test_scaled)
y_proba = xgb_classifier.predict_proba(X_test_scaled)[:, 1] # Probability of class 1

# 5. Evaluate and report results
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("\n--- RESULTS: Breast Cancer Classification (BONUS) ---")
print(f"XGBoost Model Parameters:")
print(f"  - Estimators: {xgb_classifier.n_estimators}")
print(f"  - Learning Rate: {xgb_classifier.learning_rate}")
print(f"  - Max Depth: {xgb_classifier.max_depth}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC AUC Score: {roc_auc:.4f}")
print("---------------------------------------------------------")


Starting XGBoost Classification Training (BONUS)...
Training Complete.

--- RESULTS: Breast Cancer Classification (BONUS) ---
XGBoost Model Parameters:
  - Estimators: 100
  - Learning Rate: 0.1
  - Max Depth: 3
Test Accuracy: 0.9591
Test ROC AUC Score: 0.9957
---------------------------------------------------------
